[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/12_linear_attention.ipynb)

# 🔴 Hard: Linear Self-Attention

Implement **Linear Attention** — O(S·D²) instead of O(S²·D), enabling efficient long-sequence processing.

Replace softmax with a **kernel feature map** $\phi$:

$$\text{LinearAttn}(Q,K,V) = \frac{\phi(Q) \left(\phi(K)^T V\right)}{\phi(Q) \cdot \sum \phi(K)}$$

### Feature map
Use $\phi(x) = \text{elu}(x) + 1$ (ensures non-negative features).

### Signature
```python
def linear_attention(Q, K, V):
    # Q: (B, S, D_k), K: (B, S, D_k), V: (B, S, D_v)
    # Returns: (B, S, D_v)
```

### Key insight
Instead of computing the $S \times S$ attention matrix, compute $\phi(K)^T V$ first (a $D_k \times D_v$ matrix), then multiply by $\phi(Q)$.

### Rules
- Must use a feature map (NOT softmax)
- Must be O(S·D²) — should run fast on long sequences
- You **may** use `F.elu`

In [1]:
import torch
import torch.nn.functional as F

In [16]:
# ✏️ YOUR IMPLEMENTATION HERE
def phi(x):
    return F.elu(x)+1

def linear_attention(Q, K, V):
    phi_Q,phi_K = phi(Q),phi(K)
    kv_rep = torch.einsum('...sk,...sv->...kv',phi_K,phi(V))
    numerator = torch.einsum('...sk,...kv->...sv',phi_Q,kv_rep)
    sum_phi_k = torch.sum(phi_K,dim=-2) # summation over the sequence
    denominator = torch.einsum('...sk,...k->...s',phi_Q,sum_phi_k).unsqueeze(-1)
    return numerator/denominator

In [17]:
# 🧪 Debug
Q = torch.randn(1, 8, 16)
K = torch.randn(1, 8, 16)
V = torch.randn(1, 8, 32)
out = linear_attention(Q, K, V)
print("Output shape:", out.shape)   # (1, 8, 32)
print("Has NaN?", torch.isnan(out).any().item())

Output shape: torch.Size([1, 8, 32])
Has NaN? False


In [18]:
from torch_judge import check
check('linear_attention')


🧪 Testing: Linear Self-Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (8.6ms)
  ✅ [2/4] No NaN or Inf (4.8ms)
  ✅ [3/4] Gradient flow (18.2ms)
  ✅ [4/4] Runs fast on long sequences (linear complexity) (78.7ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (110.3ms total)
  Progress saved. Run status() to see your dashboard.

